# Synthetic TTS Training Dataset Generator

Generates LJSpeech-style TTS training datasets from text prompts using a **single configurable donor TTS** voice and optional voice conversion.  
When a folder of reference voices is provided, **one complete LJSpeech dataset is produced per reference voice**.

You can use metadata.csv from an existing TTS dataset or find .txt prompts here:
  - https://huggingface.co/datasets/fdemelo/phonetic-piper-recording-studio-prompts
  - https://huggingface.co/collections/TigreGotico/synthetic-tts-datasets
  - https://github.com/rhasspy/piper-recording-studio/tree/master/prompts

### Pipeline stages

| # | Stage | Description |
|---|---|---|
| 0 | Configuration | Set all paths, parameters, and flags via env vars |
| 1 | Load Prompts | Read sentences from `prompts.txt`, a folder of `.txt` files, or `metadata.csv` |
| 2 | Donor TTS Synthesis | Single configured TTS engine + voice generates all prompts once |
| 3 | Post-Processing | Optional audio super-resolution, silence trimming, volume normalisation |
| 4 | Voice Conversion + Dataset Assembly | For each reference voice: VC all utterances → build LJSpeech dataset (`filename|transcription`) |
| 5 | Quality Filter | Score every clip and refuse the ones that fail the pass criteria |
| 6 | Upload to HuggingFace | Optional push to HF (one repo per voice) |
| 7 | Output Summary | File counts and sanity checks |

### Output structure
```
base_dir/
  1_donor_tts/           # Raw donor TTS audio (synthesised once)
  2_processed/           # Post-processed donor audio
  3_datasets/
    voice_alice/         # ← One per reference voice
      wavs/
        LJ001-0001.wav
      refused/           # ← Clips a quality rule refused (moved, never deleted)
      metadata.csv       # filename|transcription (kept clips only)
      filter_report.csv  # every metric and the refusal reason, one row per clip
    donor_raw/            # ← If no VC
      wavs/
      metadata.csv
```

### Usage
1. Edit the configuration cell below.
2. Run cells sequentially.
3. Each stage is self-contained — pip deps are installed per step.
4. Existing output files are skipped on re-run (resume-safe).

---

> **Credits:** Based on pipelines from [TigreGotico/synthetic_dataset_generator](https://github.com/TigreGotico/synthetic_dataset_generator).  
> Funded through the [NGI0 Commons Fund](https://nlnet.nl/commonsfund) via [NLnet](https://nlnet.nl), with support from the European Commission's Next Generation Internet programme (grant No 101135429).

> ⚠ **NOT EXECUTED** — This notebook requires GPU/heavy dependencies.
> It is validated for parse correctness by CI but not executed automatically.
> Run it on Kaggle (free T4) or Google Colab (free A100).


---
## NGI0 Grant Attribution

> Developed by TigreGotico for OpenVoiceOS, funded by the
> [NGI0 Commons Fund](https://nlnet.nl/project/OpenVoiceOS) / NLnet grant **101135429**.
> European Commission Next Generation Internet programme — DG CONNECT,
> grant agreement No 101135429.
> Additional funding: Swiss State Secretariat for Education, Research and Innovation (SERI).

**Origin:** `TigreGotico/ml-notebooks` `tts/tts_dataset_gen.ipynb`  
Continues the archived `TigreGotico/synthetic_dataset_generator`.  
**NGI work item:** WP3 — Synthetic TTS training dataset generator  
**CI execution:** ❌ parse-only — see banner above  

---


---
## 0 · Global Configuration

All parameters are set as environment variables via `os.environ.setdefault()`.  
Override any of them from your shell, a `.env` file, or `%env` magic **before** running the config cell.

**Edit the values below to match your setup, then run the cell.**

In [ ]:
import os

def _default_base_dir(name):
    """Somewhere writable that survives the session on the current platform."""
    if os.path.isdir("/kaggle"):
        return f"/kaggle/working/{name}"
    if os.path.isdir("/content"):
        return f"/content/{name}"
    return f"./{name}"


# ──────────────────────────────────────────────
# GENERAL
# ──────────────────────────────────────────────
os.environ.setdefault("TTS_LANG",            "en")                 # Language code (en, es, pt, …)
os.environ.setdefault("TTS_BASE_DIR",        _default_base_dir("tts_dataset"))  # Root directory for all I/O
os.environ.setdefault("TTS_DATASET_NAME",    "my_tts_dataset")     # Base name for output datasets

# ──────────────────────────────────────────────
# INPUT SOURCES  (Stage 1)
#   Set exactly ONE of these three:
#   - PROMPTS_FILE   : plain text (one sentence per line) or
#                      piper-recording-studio format (sentence_id<TAB>text)
#   - PROMPTS_DIR    : folder of .txt files (each line = one sentence)
#   - PROMPTS_CSV    : CSV/TSV with a 'text' or 'transcript' column
# ──────────────────────────────────────────────
os.environ.setdefault("PROMPTS_FILE",        "prompts.txt")        # Path to prompts.txt
os.environ.setdefault("PROMPTS_DIR",         "")                   # Path to folder of .txt files
os.environ.setdefault("PROMPTS_CSV",         "")                   # Path to metadata.csv
os.environ.setdefault("CSV_TEXT_COL",        "text")               # Column name for text in CSV
os.environ.setdefault("CSV_SEP",             "|")                  # CSV separator (| for LJSpeech style)

# ──────────────────────────────────────────────
# DONOR TTS  (Stage 2)
#   A single TTS engine + voice used to synthesise all prompts.
#   Supported engines: phoonnx, edge, google
#
#   Phoonnx       : ONNX runtime TTS (supports all piper voices + more).
#                   The preferred donor: use a phoonnx voice for the
#                   configured language wherever one exists.
#                   "default" resolves the language's default voice.
#                   examples: OpenVoiceOS/phoonnx_pt-PT_miro_tugaphone
#                   repo: https://github.com/TigreGotico/phoonnx
#   Edge          : falls back for languages with no phoonnx voice.
#                   `python -c "from ovos_tts_plugin_edge_tts import VOICES; print(VOICES)"`
#                   examples: en-US-JennyNeural, en-US-GuyNeural, en-GB-SoniaNeural
#   Google        : voice is ignored, only lang matters
# ──────────────────────────────────────────────
os.environ.setdefault("DONOR_ENGINE",        "phoonnx")            # phoonnx | edge | google
os.environ.setdefault("DONOR_VOICE",         "default")            # Voice identifier for chosen engine
os.environ.setdefault("DONOR_RATE",          "+0%")                # Speech rate (Edge only, e.g. +10%, -5%)
os.environ.setdefault("DONOR_SLOW",          "0")                  # Google only: 1 = slow speed

# ── Phoonnx-specific options (only when DONOR_ENGINE=phoonnx) ────
os.environ.setdefault("PHOONNX_PHONETIC_SPELL", "1")               # Fix pronunciations for some words/langs
os.environ.setdefault("PHOONNX_NOISE_SCALE",    "0.667")           # Generator noise
os.environ.setdefault("PHOONNX_LENGTH_SCALE",   "1.0")             # Phoneme length (>1 = slower speech)
os.environ.setdefault("PHOONNX_NOISE_W",        "0.8")             # Phoneme width noise
os.environ.setdefault("PHOONNX_DIACRITICS",     "0")               # Add diacritics (Arabic/Hebrew only)

# ──────────────────────────────────────────────
# VOICE CONVERSION  (Stage 4)
#   A folder of reference voice WAVs.  One LJSpeech dataset is
#   generated per reference voice file.
#   Leave REF_VOICES_DIR empty to produce a single dataset from
#   the raw donor TTS audio (no voice conversion).
# ──────────────────────────────────────────────
os.environ.setdefault("REF_VOICES_DIR",      "")                   # Folder of reference voice WAVs
os.environ.setdefault("VC_ENGINE",           "chatterbox")         # voiceclonnx engine alias; `voiceclonnx list` names the rest

# ──────────────────────────────────────────────
# ONNX RUNTIME  (Stages 2, 4, 5)
#   Every ONNX model in this notebook — the phoonnx donor, the voiceclonnx
#   voice converter, the speakeronnx/speechonnxmetrics scorers — is created
#   through the same provider selection.
# ──────────────────────────────────────────────
os.environ.setdefault("ONNX_PROVIDERS",      "auto")               # comma list of onnxruntime providers, or "auto"
os.environ.setdefault("ONNX_RUNTIME",        "cpu")                # cpu | cuda | rocm — which onnxruntime build the install cells fetch

# ONNX_PROVIDERS feeds phoonnx directly through its own env var.
os.environ["PHOONNX_ONNX_PROVIDERS"] = os.environ["ONNX_PROVIDERS"]

# ──────────────────────────────────────────────
# POST-PROCESSING  (Stage 3)
# ──────────────────────────────────────────────
os.environ.setdefault("TARGET_SR",           "24000")              # Final sample rate (24000, the voiceclonnx chatterbox engine's native output rate)
os.environ.setdefault("TARGET_DBFS",         "-20.0")              # Volume normalisation target
os.environ.setdefault("MIN_DURATION",        "0.5")                # Seconds — discard shorter clips
os.environ.setdefault("MAX_DURATION",        "15.0")               # Seconds — discard longer clips
os.environ.setdefault("TRIM_SILENCE",        "1")                  # 1 = trim leading/trailing silence
os.environ.setdefault("TRIM_TOP_DB",         "30")                 # dB threshold for silence trimming

# ── Audio Super-Resolution/Resampling ─
#   Upscales ~16 kHz TTS audio to 48 kHz before final resample.
#   Leave SR_ENGINE empty to use plain librosa resampling.
#
#   None of the three enhancers is published on PyPI, so each stays optional:
#   when the package is missing, super-resolution is skipped and plain librosa
#   resampling is used instead.
#
#   novasr  : 52 kB model, ~3500x RT on GPU. 16→48 kHz.
#             https://github.com/ysharma3501/NovaSR
#   flashsr : ~500 kB ONNX, ~200-400x RT. 16→48 kHz. CPU-friendly (slow).
#             https://github.com/ysharma3501/FlashSR
#   lavasr  : Vocos-based speech enhancement + upsampling.
#             https://github.com/ysharma3501/LavaSR
# ──────────────────────────────────────────────
os.environ.setdefault("SR_ENGINE",           "")                    # novasr | flashsr | lavasr | (empty=off)
os.environ.setdefault("SR_DEVICE",           "cuda")               # cuda | cpu - TODO: work in progress, not used
os.environ.setdefault("LAVASR_DENOISE",      "1")                  # 1 = enable LavaSR denoiser
os.environ.setdefault("LAVASR_ENHANCE",      "1")                  # 1 = enable LavaSR denoiser

# ──────────────────────────────────────────────
# QUALITY FILTER  (Stage 5)
#   Scores every clip built in Stage 4 and refuses the ones that fail
#   the pass criteria.  A refused wav is moved to a sibling refused/
#   directory, never deleted.  Leave SPK_MIN_COS, QUALITY_BARS or
#   STT_MODULE empty to score and report that criterion without
#   refusing on it.
# ──────────────────────────────────────────────
os.environ.setdefault("SPK_MIN_COS",         "")                   # Minimum speaker cosine (empty = report only)
os.environ.setdefault("VC_LEN_TOL_S",        "0.06")               # Maximum |converted - source| duration in seconds (measured: voiceclonnx chatterbox lengthens clips by up to 0.056 s)
os.environ.setdefault("ECHO_THRESH",         "0.85")               # Envelope-cosine threshold for a reference echo
os.environ.setdefault("CER_MAX",             "0.10")               # Maximum character error rate on the round trip
os.environ.setdefault("QUALITY_FAMILIES",    "utmos,dnsmos")       # speechonnxmetrics families to score
os.environ.setdefault("QUALITY_BARS",        "{}")                 # JSON quality floor, e.g. {"utmos": 3.0} (empty = report only)
os.environ.setdefault("READ_WER_MAX",        "")                   # Maximum word error rate between the clip and source readings (empty = report only)
os.environ.setdefault("STT_MODULE",          "")                   # OVOS STT plugin for the round trip (empty = skip)
os.environ.setdefault("STT_CONFIG",          "{}")                 # JSON plugin config for STT_MODULE

# ──────────────────────────────────────────────
# HUGGINGFACE UPLOAD  (Stage 6)
# ──────────────────────────────────────────────
os.environ.setdefault("HF_UPLOAD",           "0")                  # 1 = push to HuggingFace
os.environ.setdefault("HF_REPO_PREFIX",      "")                   # e.g. your-user/tts — voice name is appended
os.environ.setdefault("HF_TOKEN",            "")                   # HF write token (or use huggingface-cli login)

# ──────────────────────────────────────────────
# Derived paths
# ──────────────────────────────────────────────
BASE = os.environ["TTS_BASE_DIR"]
LANG = os.environ["TTS_LANG"]
NAME = os.environ["TTS_DATASET_NAME"]

os.environ.setdefault("DIR_DONOR_TTS",       os.path.join(BASE, "1_donor_tts"))
os.environ.setdefault("DIR_PROCESSED",       os.path.join(BASE, "2_processed"))
os.environ.setdefault("DIR_DATASETS",        os.path.join(BASE, "3_datasets"))


def _env(key, default=""):
    return os.environ.get(key, default)


def _flag(key):
    return _env(key, "0").strip().lower() in ("1", "true", "yes")


print("Configuration loaded.")
print(f"  Language       : {LANG}")
print(f"  Dataset name   : {NAME}")
print(f"  Base directory : {BASE}")
print(f"  Donor TTS      : {_env('DONOR_ENGINE')} / {_env('DONOR_VOICE')}")
if _env('DONOR_ENGINE') == 'phoonnx':
    print(f"    phonetic_spell={_flag('PHOONNX_PHONETIC_SPELL')}  noise_scale={_env('PHOONNX_NOISE_SCALE')}  "
          f"length_scale={_env('PHOONNX_LENGTH_SCALE')}  noise_w={_env('PHOONNX_NOISE_W')}  "
          f"diacritics={_flag('PHOONNX_DIACRITICS')}")
sr_engine = _env('SR_ENGINE')
if sr_engine:
    print(f"  Super-res      : {sr_engine} on {_env('SR_DEVICE')}")
else:
    print(f"  Super-res      : off (librosa resample to {_env('TARGET_SR')} Hz)")
ref_dir = _env('REF_VOICES_DIR')
if ref_dir:
    from pathlib import Path as _P
    n_refs = len(list(_P(ref_dir).rglob('*.wav'))) if _P(ref_dir).is_dir() else 0
    print(f"  VC enabled     : yes ({n_refs} reference voices in {ref_dir})")
    print(f"  VC engine      : {_env('VC_ENGINE')}")
else:
    print(f"  VC enabled     : no (single donor-only dataset)")
print(f"  ONNX providers : {_env('ONNX_PROVIDERS')}  (onnxruntime build: {_env('ONNX_RUNTIME')})")
print(f"  Quality filter : SPK_MIN_COS={_env('SPK_MIN_COS') or '(report only)'}  "
      f"VC_LEN_TOL_S={_env('VC_LEN_TOL_S', '0.06')}  "
      f"CER_MAX={_env('CER_MAX', '0.10')}  ECHO_THRESH={_env('ECHO_THRESH', '0.85')}  "
      f"round_trip={'on (' + _env('STT_MODULE') + ')' if _env('STT_MODULE') else 'off'}")
print(f"  HF upload      : {'yes' if _flag('HF_UPLOAD') else 'no'}")

---
## 1 · Load Prompts

Reads text prompts from one of three sources (in priority order):
1. `PROMPTS_CSV` — a CSV/TSV with a text column (pipe-separated LJSpeech style supported)
2. `PROMPTS_DIR` — a folder of `.txt` files, each containing one sentence per line
3. `PROMPTS_FILE` — a single text file with one sentence per line, or piper-recording-studio format (`sentence_id\ttext`)

Output is a deduplicated list of sentences stored in `PROMPTS`.

In [ ]:
import os
import re
from pathlib import Path


def _clean_text(text):
    """Strip and collapse whitespace.  Returns empty string for junk."""
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    return text if len(text) > 1 else ""


def load_prompts_from_file(path):
    """One sentence per line, or piper-recording-studio TSV: sentence_id<TAB>text."""
    lines = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            # piper-recording-studio format: sentence_id<TAB>text
            if "\t" in line:
                parts = line.split("\t", 1)
                line = parts[-1]  # discard sentence_id, keep text
            t = _clean_text(line)
            if t:
                lines.append(t)
    return lines


def load_prompts_from_dir(folder):
    """Read every .txt file in folder, one sentence per line."""
    lines = []
    for txt in sorted(Path(folder).rglob("*.txt")):
        lines.extend(load_prompts_from_file(txt))
    return lines


def load_prompts_from_csv(path, text_col, sep):
    """Read a CSV; auto-detects column by name or position."""
    import csv
    lines = []
    with open(path, "r", encoding="utf-8") as f:
        # Sniff whether there's a header
        sample = f.read(4096)
        f.seek(0)
        has_header = csv.Sniffer().has_header(sample)
        reader = csv.reader(f, delimiter=sep)

        if has_header:
            header = next(reader)
            header_lower = [h.strip().lower() for h in header]
            col_idx = None
            for candidate in [text_col.lower(), "text", "transcript", "transcription",
                              "sentence"]:
                if candidate in header_lower:
                    col_idx = header_lower.index(candidate)
                    break
            if col_idx is None:
                # LJSpeech: filename|text — text is column 1 or last
                col_idx = min(1, len(header) - 1)
                print(f"  Warning: text column '{text_col}' not found in header {header}.")
                print(f"           Falling back to column index {col_idx}.")
        else:
            # No header — assume LJSpeech: col 0 = filename, col 1 = text
            col_idx = 1 if sep == "|" else 0

        for row in reader:
            if len(row) <= col_idx:
                continue
            t = _clean_text(row[col_idx])
            if t:
                lines.append(t)
    return lines


# ── Load from the configured source ──────────────────────────────────
csv_path  = _env("PROMPTS_CSV")
dir_path  = _env("PROMPTS_DIR")
file_path = _env("PROMPTS_FILE")

raw_prompts = []
if csv_path and os.path.exists(csv_path):
    print(f"Loading prompts from CSV: {csv_path}")
    raw_prompts = load_prompts_from_csv(csv_path, _env("CSV_TEXT_COL", "text"), _env("CSV_SEP", "|"))
elif dir_path and os.path.isdir(dir_path):
    print(f"Loading prompts from directory: {dir_path}")
    raw_prompts = load_prompts_from_dir(dir_path)
elif file_path and os.path.exists(file_path):
    print(f"Loading prompts from file: {file_path}")
    raw_prompts = load_prompts_from_file(file_path)
else:
    print("ERROR: No valid prompt source found.")
    print("  Set one of: PROMPTS_FILE, PROMPTS_DIR, or PROMPTS_CSV")

# Deduplicate while preserving order
seen = set()
PROMPTS = []
for text in raw_prompts:
    key = text.lower().strip()
    if key not in seen:
        seen.add(key)
        PROMPTS.append(text)

print(f"  Loaded {len(raw_prompts)} lines -> {len(PROMPTS)} unique prompts")
if PROMPTS:
    print(f"  First: {PROMPTS[0][:80]}{'…' if len(PROMPTS[0]) > 80 else ''}")
    print(f"  Last : {PROMPTS[-1][:80]}{'…' if len(PROMPTS[-1]) > 80 else ''}")

---
## 2 · Donor TTS Synthesis

Uses the **single configured donor voice** to synthesise every prompt exactly once.  
This is the "master" audio that gets voice-converted into each target voice in Stage 4.

| Key env vars | |
|---|---|
| `DONOR_ENGINE` | `phoonnx` / `edge` / `google` |
| `DONOR_VOICE` | Voice identifier (engine-specific); `default` for phoonnx's per-language default |
| `DONOR_RATE` | Speech rate override (Edge only) |
| `PHOONNX_*` | Phoonnx synthesis parameters |
| `ONNX_PROVIDERS` | onnxruntime execution providers for the phoonnx voice (see Stage 0) |

In [ ]:
donor_engine = _env("DONOR_ENGINE", "phoonnx")
onnx_runtime = _env("ONNX_RUNTIME", "cpu").strip().lower()

if donor_engine == "edge":
    # Edge TTS needs outbound internet access.
    !pip install --quiet --break-system-packages --pre "ovos_tts_plugin_edge_tts>=0.3.2a2" "nest_asyncio>=1.6"

    # patch so edge-TTS works in notebooks
    import nest_asyncio
    nest_asyncio.apply()

elif donor_engine == "google":
    !pip install --quiet --break-system-packages --pre "ovos_tts_plugin_google_tx>=1.0.5a4"
elif donor_engine == "phoonnx":
    !pip install --quiet --break-system-packages --pre "phoonnx>=1.91.1a1" "ovos-utils>=0.14.1a1"
    if onnx_runtime == "cuda":
        !pip install --quiet --break-system-packages --pre "onnxruntime-gpu>=1.18"
    elif onnx_runtime == "rocm":
        print("AMD ROCm: install onnxruntime-rocm or the MIGraphX build from AMD's package "
              "index yourself; nothing is installed here and CPU is the fallback.")
    else:
        !pip install --quiet --break-system-packages --pre "onnxruntime>=1.18"
!pip install --quiet --break-system-packages --pre "tqdm>=4.66"

In [ ]:
import os
import sys
import json
import time
import warnings
from pathlib import Path
from tqdm import tqdm

warnings.filterwarnings("ignore")


# ═══════════════════════════════════════════════════════════════════
# Instantiate the single donor TTS plugin
# ═══════════════════════════════════════════════════════════════════

def _make_donor_plugin(engine: str, voice: str, lang: str, rate: str, slow: bool):
    """
    Create a single TTS plugin instance from top-level config.
    Returns (plugin, voice_id) where voice_id is used for filenames.
    """
    if engine == "edge":
        from ovos_tts_plugin_edge_tts import EdgeTTSPlugin
        cfg = {"voice": voice}
        if rate:
            cfg["rate"] = rate
        return EdgeTTSPlugin(config=cfg), f"edge_{voice}"

    if engine == "google":
        from ovos_tts_plugin_google_tx import GoogleTranslateTTS
        cfg = {"lang": lang, "slow": slow}
        return GoogleTranslateTTS(config=cfg), f"google_{lang}"

    if engine == "phoonnx":
        from phoonnx.opm import PhoonnxTTSPlugin
        cfg = {
            "lang": lang,
            "enable_phonetic_spelling": _flag("PHOONNX_PHONETIC_SPELL"),
            "noise-scale":  float(_env("PHOONNX_NOISE_SCALE", "0.667")),
            "length-scale": float(_env("PHOONNX_LENGTH_SCALE", "1.0")),
            "noise-w":      float(_env("PHOONNX_NOISE_W", "0.8")),
            "add_diacritics": _flag("PHOONNX_DIACRITICS"),
        }
        if voice and voice != "default":
            cfg["voice"] = voice
        return PhoonnxTTSPlugin(config=cfg), f"phoonnx_{voice}"

    raise ValueError(f"Unknown donor engine: {engine}")


# ═══════════════════════════════════════════════════════════════════
# Synthesis loop — one WAV per prompt, single donor voice
# ═══════════════════════════════════════════════════════════════════

def synthesize_with_donor(
    prompts: list,
    plugin,
    voice_id: str,
    lang: str,
    output_dir: str,
):
    """
    Synthesise every prompt with the donor plugin.  Saves WAVs as
    {idx:06d}.wav plus a JSONL manifest for traceability.
    Resume-safe: skips prompts whose output WAV already exists.
    """
    os.makedirs(output_dir, exist_ok=True)
    manifest_path = os.path.join(output_dir, "tts_manifest.jsonl")

    # Load existing manifest entries for resume
    existing = {}
    if os.path.exists(manifest_path):
        with open(manifest_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    rec = json.loads(line)
                    existing[rec["idx"]] = rec
        print(f"  Resuming: {len(existing)} already synthesised.")

    manifest_f = open(manifest_path, "a", encoding="utf-8")
    success, fail, skipped = 0, 0, 0
    result_map = dict(existing)
    start = time.time()

    for idx, text in enumerate(tqdm(prompts, desc="Donor TTS", unit="utt", file=sys.stdout)):
        wav_name = f"{idx:06d}.wav"
        wav_path = os.path.join(output_dir, wav_name)

        if idx in existing and os.path.exists(wav_path):
            skipped += 1
            continue

        try:
            plugin.get_tts(text, wav_path, lang=lang)
            rec = {
                "idx": idx,
                "wav": wav_name,
                "text": text,
                "engine": _env("DONOR_ENGINE"),
                "voice": voice_id,
            }
            manifest_f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            manifest_f.flush()
            result_map[idx] = rec
            success += 1
            #print("Generated:", wav_name)
        except Exception as e:
            tqdm.write(f"  Failed #{idx}: {e}")
            fail += 1
            if fail > 10:
                raise ValueError("TTS synthesis failed 10 times, aborting.")

    manifest_f.close()
    elapsed = time.time() - start
    print(f"  Done: {success} new, {skipped} skipped, {fail} failed  ({elapsed:.1f}s)")
    return result_map


# ═══════════════════════════════════════════════════════════════════
# Run donor TTS
# ═══════════════════════════════════════════════════════════════════

if not PROMPTS:
    raise RuntimeError("No prompts loaded — run Stage 1 first.")
else:
    engine   = _env("DONOR_ENGINE", "phoonnx")
    voice    = _env("DONOR_VOICE", "default")
    rate     = _env("DONOR_RATE", "+0%")
    slow     = _flag("DONOR_SLOW")

    print(f"Donor: {engine} / {voice}")
    donor_plugin, donor_voice_id = _make_donor_plugin(engine, voice, LANG, rate, slow)

    donor_dir = _env("DIR_DONOR_TTS")
    TTS_MANIFEST = synthesize_with_donor(
        prompts=PROMPTS,
        plugin=donor_plugin,
        voice_id=donor_voice_id,
        lang=LANG,
        output_dir=donor_dir,
    )
    print(f"\nDonor TTS complete.  {len(TTS_MANIFEST)} utterances in {donor_dir}")

---
## 3 · Post-Processing

For each donor WAV:
1. **Audio super-resolution** (optional) — upscale ~16 kHz TTS to 48 kHz using NovaSR, FlashSR, or LavaSR.
   NovaSR and LavaSR have no PyPI release: unless you installed them yourself the stage is skipped
   and plain resampling is used.
2. Resample to target sample rate (default 24000 Hz, the voiceclonnx chatterbox engine's native output rate)
3. Convert to mono
4. Trim leading/trailing silence
5. Normalise volume to target dBFS
6. Filter by min/max duration

When `SR_ENGINE` is set, super-resolution replaces the initial resample — the SR model
upscales to 48 kHz, then a final `librosa.resample(48000 → TARGET_SR)` brings it to the
desired output rate.

| Key env vars | |
|---|---|
| `SR_ENGINE` | `novasr` / `flashsr` / `lavasr` / empty=off |
| `SR_DEVICE` | `cuda` / `cpu` (novasr/lavasr) |
| `LAVASR_DENOISE` | `1` = enable LavaSR denoiser |
| `TARGET_SR` | Final sample rate (24000) |
| `TARGET_DBFS` | Volume target in dBFS |
| `MIN_DURATION` / `MAX_DURATION` | Duration filter (seconds) |
| `TRIM_SILENCE` | 1 = trim silence |
| `TRIM_TOP_DB` | Silence threshold (dB) |

In [ ]:
sr_engine = _env("SR_ENGINE").strip().lower()

if sr_engine == "flashsr":
    !pip install --quiet --break-system-packages --pre "onnxruntime>=1.18" "numpy>=1.26" "huggingface_hub>=0.34"
elif sr_engine in ("novasr", "lavasr"):
    # NovaSR and LavaSR are not on PyPI. Install them yourself from their own
    # releases if you want super-resolution; the stage is skipped otherwise.
    import importlib.util
    _mod = "NovaSR" if sr_engine == "novasr" else "LavaSR"
    if importlib.util.find_spec(_mod) is None:
        print(f"{_mod} is not installed and has no PyPI release — "
              f"super-resolution is skipped, plain librosa resampling is used.")
        sr_engine = ""
        os.environ["SR_ENGINE"] = ""

!pip install --quiet --break-system-packages --pre "librosa>=0.10.2" "soundfile>=0.12.1" "numpy>=1.26" "tqdm>=4.66"

In [ ]:
import os
import sys
import numpy as np
import librosa
import soundfile as sf
from pathlib import Path
from tqdm import tqdm

import onnxruntime as ort


# ═══════════════════════════════════════════════════════════════════
# Load SR model once (if enabled)
# ═══════════════════════════════════════════════════════════════════

def _load_sr_model(engine: str, device: str):
    """Load and return the super-resolution model + its output sample rate."""
    if engine == "novasr":
        from NovaSR import FastSR
        model = FastSR()
        return model, 48000

    if engine == "flashsr":
        from huggingface_hub import hf_hub_download
        # Download the ONNX model from HF Hub
        model_path = hf_hub_download(repo_id="YatharthS/FlashSR", filename="model.onnx", subfolder="onnx")
        ort_session = ort.InferenceSession(model_path)
        return ort_session, 48000

    if engine == "lavasr":
        from LavaSR.model import LavaEnhance
        model = LavaEnhance("YatharthS/LavaSR", device)
        return model, 48000

    raise ValueError(f"Unknown SR engine: {engine}")


def _apply_sr(wav_np: np.ndarray, sr_in: int, engine: str, model, sr_out: int) -> tuple:
    """
    Apply super-resolution to a numpy audio array.
    Returns (upsampled_wav, new_sample_rate).
    """
    import torch

    # Load audio file at 16kHz
    if sr_in != 16000:
        wav_np = librosa.resample(wav_np, orig_sr=sr_in, target_sr=16000)  # Resamples to 16kHz

    if engine == "novasr":
        # NovaSR expects 16 kHz input loaded via its own loader, but also works with raw tensors.
        lowres = torch.from_numpy(wav_np).unsqueeze(0).unsqueeze(1).half().to(model.device)
        #highres = model.infer(lowres).cpu().numpy().squeeze()
        highres = model.infer(lowres).cpu().detach().numpy().squeeze().astype(np.float32)
        return highres, 48000

    if engine == "flashsr":
        lowres_wav = wav_np[np.newaxis, :]  # Add batch dimension
        onnx_output = model.run(["reconstruction"], {"audio_values": lowres_wav})[0]
        highres = onnx_output.squeeze(0)
        return highres, 48000

    if engine == "lavasr":
        lowres = torch.from_numpy(wav_np).unsqueeze(0)
        denoise = _flag("LAVASR_DENOISE")
        enhance = _flag("LAVASR_DENOISE")
        highres = model.enhance(lowres, enhance=enhance, denoise=denoise).cpu().numpy().squeeze()
        return highres, 48000  # docs say LavaSR outputs at 16 kHz enhanced but that is wrong...

    # Should not reach here
    return wav_np, sr_in


# ═══════════════════════════════════════════════════════════════════
# Main post-processing loop
# ═══════════════════════════════════════════════════════════════════

def postprocess_audio(
    manifest: dict,
    source_dir: str,
    output_dir: str,
    target_sr: int = 24000,
    target_dbfs: float = -20.0,
    min_dur: float = 0.5,
    max_dur: float = 15.0,
    trim: bool = True,
    trim_db: int = 30,
    sr_engine: str = "",
    sr_model=None,
    sr_model_out_rate: int = 48000,
):
    """
    Post-process each donor WAV: optional SR, resample, mono, trim, normalise, filter.
    Returns a filtered manifest of successfully processed utterances.
    Resume-safe: skips files that already exist in output_dir.
    """
    os.makedirs(output_dir, exist_ok=True)
    kept, dropped, skipped = 0, 0, 0
    filtered_manifest = {}

    for idx in tqdm(sorted(manifest.keys()), desc="Post-processing", unit="utt", file=sys.stdout):
        rec = manifest[idx]
        src_path = os.path.join(source_dir, rec["wav"])
        dst_path = os.path.join(output_dir, rec["wav"])

        # Resume check
        if os.path.exists(dst_path):
            try:
                info = sf.info(dst_path)
                dur = info.frames / info.samplerate
                if min_dur <= dur <= max_dur:
                    filtered_manifest[idx] = rec
                    skipped += 1
                    continue
            except Exception:
                pass  # re-process

        if not os.path.exists(src_path):
            dropped += 1
            continue

        try:
            # Load at native SR first (for SR models that want 16k input)
            wav, sr = librosa.load(src_path, sr=None, mono=True)

            # Audio super-resolution (if enabled)
            if sr_engine and sr_model is not None:
                wav, sr = _apply_sr(wav, sr, sr_engine, sr_model, sr_model_out_rate)

            # Resample to final target SR
            if sr != target_sr:
                wav = librosa.resample(wav, orig_sr=sr, target_sr=target_sr)
                sr = target_sr

            if trim:
                wav, _ = librosa.effects.trim(wav, top_db=trim_db)

            dur = len(wav) / sr
            if dur < min_dur or dur > max_dur:
                dropped += 1
                continue

            # Volume normalisation
            rms = np.sqrt(np.mean(wav ** 2) + 1e-9)
            current_db = 20 * np.log10(rms + 1e-9)
            gain_linear = 10 ** ((target_dbfs - current_db) / 20.0)
            wav = np.clip(wav * gain_linear, -1.0, 1.0)

            sf.write(dst_path, wav, sr)
            filtered_manifest[idx] = rec
            kept += 1

        except Exception as e:
            tqdm.write(f"  Error processing {rec['wav']}: {e}")
            dropped += 1
            if dropped > 10:
                #  novasr - Input type (float) and bias type (c10::Half) should be the same
                raise ValueError("Failed to process 10 wavs. aborting")

    print(f"  Result: {kept} new + {skipped} existing = {kept + skipped} kept, {dropped} dropped")
    return filtered_manifest


# ═══════════════════════════════════════════════════════════════════
# Run post-processing
# ═══════════════════════════════════════════════════════════════════

sr_engine_name = _env("SR_ENGINE").strip().lower()
sr_model_obj, sr_out_rate = None, 48000

if sr_engine_name:
    print(f"Loading SR model: {sr_engine_name} on {_env('SR_DEVICE', 'cpu')} ...")
    sr_model_obj, sr_out_rate = _load_sr_model(sr_engine_name, _env("SR_DEVICE", "cpu"))
    print("  SR model loaded.")

PROCESSED_MANIFEST = postprocess_audio(
    manifest=TTS_MANIFEST,
    source_dir=_env("DIR_DONOR_TTS"),
    output_dir=_env("DIR_PROCESSED"),
    target_sr=int(_env("TARGET_SR", "24000")),
    target_dbfs=float(_env("TARGET_DBFS", "-20.0")),
    min_dur=float(_env("MIN_DURATION", "0.5")),
    max_dur=float(_env("MAX_DURATION", "15.0")),
    trim=_flag("TRIM_SILENCE"),
    trim_db=int(_env("TRIM_TOP_DB", "30")),
    sr_engine=sr_engine_name,
    sr_model=sr_model_obj,
    sr_model_out_rate=sr_out_rate,
)

# Free SR model memory
if sr_model_obj is not None:
    del sr_model_obj
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass

print(f"\nPost-processing complete. {len(PROCESSED_MANIFEST)} utterances ready.")

---
## 4 · Voice Conversion + LJSpeech Dataset Assembly

For **each reference voice WAV** in `REF_VOICES_DIR`:
1. Voice-convert every processed donor utterance to that voice
2. Assemble a complete LJSpeech dataset (`wavs/` + `metadata.csv`)

If `REF_VOICES_DIR` is empty, a single dataset is built directly from the donor audio (no VC).

Conversion runs through `voiceclonnx`'s `VoiceCloner`. `voiceclonnx list` prints every
engine it supports; only the default, `chatterbox`, takes an execution-provider list —
every other engine runs on CPU regardless of `ONNX_PROVIDERS`.

### Output per voice
```
3_datasets/
  {voice_stem}/
    wavs/
      LJ001-0001.wav
      ...
    metadata.csv   # filename|transcription  (no header)
```

| Key env vars | |
|---|---|
| `REF_VOICES_DIR` | Folder of reference voice WAVs |
| `VC_ENGINE` | `voiceclonnx` engine alias, default `chatterbox`; `voiceclonnx list` names the rest |
| `ONNX_PROVIDERS` | onnxruntime execution providers for the `chatterbox` engine (see Stage 0) |

In [ ]:
ref_voices_dir = _env("REF_VOICES_DIR")
vc_engine = _env("VC_ENGINE", "chatterbox")
onnx_runtime = _env("ONNX_RUNTIME", "cpu").strip().lower()

if ref_voices_dir:
    !pip install --quiet --break-system-packages --pre "voiceclonnx>=0.0.3a3" "tqdm>=4.66"
    if onnx_runtime == "cuda":
        !pip install --quiet --break-system-packages --pre "onnxruntime-gpu>=1.18"
    elif onnx_runtime == "rocm":
        print("AMD ROCm: install onnxruntime-rocm or the MIGraphX build from AMD's package "
              "index yourself; nothing is installed here and CPU is the fallback.")
    else:
        !pip install --quiet --break-system-packages --pre "onnxruntime>=1.18"
else:
    print("REF_VOICES_DIR not set — will build a single donor-only dataset.")

In [ ]:
import os
import sys
import time
import shutil
from pathlib import Path
from tqdm import tqdm


def _resolve_onnx_providers(raw: str):
    """
    A comma-separated onnxruntime provider list, or "" / "auto" to resolve
    against the installed onnxruntime build, in the same preference order
    phoonnx uses (CUDA, ROCm, MIGraphX, DirectML, CoreML, OpenVINO), CPU
    always last.
    """
    raw = (raw or "").strip()
    if not raw or raw.lower() == "auto":
        import onnxruntime as ort
        preferred = [
            "CUDAExecutionProvider", "ROCMExecutionProvider", "MIGraphXExecutionProvider",
            "DmlExecutionProvider", "CoreMLExecutionProvider", "OpenVINOExecutionProvider",
            "CPUExecutionProvider",
        ]
        available = ort.get_available_providers()
        resolved = [p for p in preferred if p in available]
        if "CPUExecutionProvider" not in resolved:
            resolved.append("CPUExecutionProvider")
        return resolved
    return [p.strip() for p in raw.split(",") if p.strip()]


def build_ljspeech_from_dir(
    manifest: dict,
    audio_dir: str,
    output_dir: str,
    chapter_prefix: str = "LJ001",
):
    """
    Copy WAVs into wavs/ with LJSpeech naming and write metadata.csv.
    Format: filename|transcription  (no header, pipe-separated).
    Also writes source_map.csv (lj_name|source_wav) — the join key the
    Quality Filter stage uses to find each clip's pre-conversion audio
    under DIR_PROCESSED.
    Returns the number of utterances written.
    """
    wavs_dir = os.path.join(output_dir, "wavs")
    os.makedirs(wavs_dir, exist_ok=True)
    meta_lines = []
    source_map_lines = []
    count = 0

    for idx in sorted(manifest.keys()):
        rec = manifest[idx]
        src = os.path.join(audio_dir, rec["wav"])
        if not os.path.exists(src):
            continue
        count += 1
        lj_name = f"{chapter_prefix}-{count:04d}"
        shutil.copy2(src, os.path.join(wavs_dir, f"{lj_name}.wav"))
        meta_lines.append(f"{lj_name}|{rec['text']}")
        source_map_lines.append(f"{lj_name}|{rec['wav']}")

    meta_path = os.path.join(output_dir, "metadata.csv")
    with open(meta_path, "w", encoding="utf-8") as f:
        f.write("\n".join(meta_lines) + "\n")

    source_map_path = os.path.join(output_dir, "source_map.csv")
    with open(source_map_path, "w", encoding="utf-8") as f:
        f.write("\n".join(source_map_lines) + "\n")

    return count


def vc_and_build_dataset(
    manifest: dict,
    processed_dir: str,
    ref_voice_path: str,
    dataset_dir: str,
    cloner,
):
    """
    Voice-convert every utterance to a single reference voice with the
    voiceclonnx cloner, then assemble a LJSpeech dataset from the results.
    Resume-safe: skips WAVs that already exist in the temp VC dir.
    Cleans up intermediate VC files after assembly to save disk.
    """
    vc_tmp = os.path.join(dataset_dir, "_vc_tmp")
    os.makedirs(vc_tmp, exist_ok=True)

    success, fail, skipped = 0, 0, 0

    for idx in tqdm(
        sorted(manifest.keys()),
        desc=f"  VC {Path(ref_voice_path).stem}",
        unit="utt",
        file=sys.stdout,
    ):
        rec = manifest[idx]
        src_path = os.path.join(processed_dir, rec["wav"])
        dst_path = os.path.join(vc_tmp, rec["wav"])

        if os.path.exists(dst_path):
            skipped += 1
            continue
        if not os.path.exists(src_path):
            fail += 1
            continue

        try:
            cloner.clone_voice(src_path, ref_voice_path, dst_path)
            success += 1
        except Exception:
            fail += 1

    print(f"    VC: {success} new, {skipped} resumed, {fail} failed")

    # Assemble LJSpeech from the VC output
    n = build_ljspeech_from_dir(manifest, vc_tmp, dataset_dir)
    print(f"    Dataset: {n} utterances -> {dataset_dir}")

    # Clean up temp VC files to save disk
    shutil.rmtree(vc_tmp, ignore_errors=True)
    return n


# ═══════════════════════════════════════════════════════════════════
# Run: iterate over reference voices (or build donor-only dataset)
# ═══════════════════════════════════════════════════════════════════

datasets_dir       = _env("DIR_DATASETS")
processed_dir      = _env("DIR_PROCESSED")
ref_voices_dir     = _env("REF_VOICES_DIR")
vc_engine          = _env("VC_ENGINE", "chatterbox")
onnx_providers_raw = _env("ONNX_PROVIDERS", "auto")

BUILT_DATASETS = {}   # voice_name -> dataset_dir
REF_VOICE_PATHS = {}  # voice_name -> reference voice wav (unset for donor_raw)

if not ref_voices_dir:
    # ── No VC: build a single dataset from processed donor audio ──
    print("No reference voices — building donor-only dataset.")
    ds_dir = os.path.join(datasets_dir, "donor_raw")
    n = build_ljspeech_from_dir(PROCESSED_MANIFEST, processed_dir, ds_dir)
    BUILT_DATASETS["donor_raw"] = ds_dir
    print(f"  Built {n} utterances -> {ds_dir}")

else:
    # ── Discover reference voices ─────────────────────────────────
    ref_files = sorted(Path(ref_voices_dir).rglob("*.wav"))
    if not ref_files:
        print(f"ERROR: No .wav files found in {ref_voices_dir}")
    else:
        print(f"Found {len(ref_files)} reference voice(s). Loading the voice cloner...")

        from voiceclonnx import VoiceCloner

        vc_kwargs = {}
        if vc_engine == "chatterbox":
            # Only the chatterbox engine takes an explicit provider list;
            # every other voiceclonnx engine runs on CPU regardless.
            vc_kwargs["providers"] = _resolve_onnx_providers(onnx_providers_raw)
        cloner = VoiceCloner(engine=vc_engine, **vc_kwargs)
        print(f"  Voice cloner loaded: engine={vc_engine}  sample_rate={cloner.sample_rate}\n")

        start_all = time.time()

        for ri, ref_path in enumerate(ref_files):
            voice_name = ref_path.stem  # e.g. "alice" from alice.wav
            ds_dir = os.path.join(datasets_dir, voice_name)
            REF_VOICE_PATHS[voice_name] = str(ref_path)

            # Skip if this dataset already looks complete
            meta_file = os.path.join(ds_dir, "metadata.csv")
            if os.path.exists(meta_file):
                existing_n = sum(1 for line in open(meta_file) if line.strip())
                if existing_n >= len(PROCESSED_MANIFEST):
                    print(f"[{ri+1}/{len(ref_files)}] {voice_name}: already complete ({existing_n} utts) — skipping.")
                    BUILT_DATASETS[voice_name] = ds_dir
                    continue

            print(f"[{ri+1}/{len(ref_files)}] Processing voice: {voice_name}")
            n = vc_and_build_dataset(
                manifest=PROCESSED_MANIFEST,
                processed_dir=processed_dir,
                ref_voice_path=str(ref_path),
                dataset_dir=ds_dir,
                cloner=cloner,
            )
            BUILT_DATASETS[voice_name] = ds_dir
            print()

        elapsed = time.time() - start_all
        print(f"\nAll voices done. {len(BUILT_DATASETS)} datasets in {elapsed:.1f}s")

print(f"\nDatasets built: {list(BUILT_DATASETS.keys())}")

---
## 5 · Quality Filter

Scores every clip in each dataset built in Stage 4 and refuses the ones that
fail the pass criteria. A refused wav is moved to a sibling `refused/`
directory, never deleted, so nothing here is unrecoverable.

Each clip is scored on:
1. **Duration**, in seconds.
2. **Speaker similarity** — cosine between the clip's speaker embedding and
   the reference voice's embedding (`speakeronnx`, WeSpeaker ResNet34).
   Skipped when the dataset has no reference voice (the `donor_raw` case).
3. **Prompt echo** — cosine between a 200-bin amplitude envelope of the clip
   and of the reference voice, each L2-normalised. A high value means the
   clip is the reference itself instead of the donor's content. The check
   does not catch a clip that carries the reference beside real content:
   such a clip scores about as low as an unrelated pair. Skipped when there
   is no reference voice.
4. **Conversion length** — converted duration minus the pre-conversion
   (donor/source) duration. On six real clips from 1.9 to 4.6 seconds,
   `voiceclonnx`'s `chatterbox` engine lengthened every one by 0.04 to
   0.056 seconds, so `VC_LEN_TOL_S` defaults to 0.06 s. The rule refuses a
   difference larger than `VC_LEN_TOL_S`, which means material was lost or
   added. The ratio is also reported. Skipped when there is no reference
   voice, since there is then no conversion to measure.
5. **Naturalness** — no-reference MOS families from `speechonnxmetrics`
   (UTMOS and DNSMOS by default).
6. **Round trip** — the clip and its pre-conversion source are both
   transcribed with the same OVOS STT plugin. CER is computed between the
   clip's transcription and the prompt text, after normalising both text
   (Unicode NFC, casefold, strip punctuation, collapse spaces). The word
   error rate between the clip's reading and the source's reading is also
   reported. Two renders of the same sentence often read differently by a
   word or two, so this rate refuses only when `READ_WER_MAX` is set; read
   its bar off the printed distribution. Skipped, and reported as skipped,
   when `STT_MODULE` is empty.

A clip is refused on the first rule it fails, checked in this order: speaker
cosine below `SPK_MIN_COS`, a length difference above `VC_LEN_TOL_S`, CER
above `CER_MAX`, a reading difference above `READ_WER_MAX`, an echo, then
any floor set in `QUALITY_BARS`. Leaving `SPK_MIN_COS`, `READ_WER_MAX`,
`QUALITY_BARS` or `STT_MODULE` empty only reports that metric — it never
refuses on it.

Voice-converted datasets are written at the active `VC_ENGINE`'s own sample
rate (`cloner.sample_rate`; 24 kHz for the default `chatterbox` engine),
whatever `TARGET_SR` is set to. `TARGET_SR` applies to the donor audio and
to the donor-only dataset. The stage prints the distribution (min, p5, p50,
p95, max) of every metric and, for echo, how many clips land within 0.05 of
`ECHO_THRESH`, so a floor can be read off the data instead of guessed.

### Output per voice
```
3_datasets/{voice}/
  wavs/               # kept clips only
  refused/            # clips a rule refused
  metadata.csv        # rewritten to the kept rows
  filter_report.csv   # every metric and the refusal reason, one row per clip
```

| Key env vars | |
|---|---|
| `SPK_MIN_COS` | Minimum speaker cosine (empty = report only) |
| `VC_LEN_TOL_S` | Maximum converted-minus-source duration, in seconds |
| `ECHO_THRESH` | Envelope-cosine threshold for a reference echo |
| `CER_MAX` | Maximum character error rate on the round trip |
| `QUALITY_FAMILIES` | `speechonnxmetrics` families to score, comma separated |
| `QUALITY_BARS` | JSON quality floor per family, e.g. `{"utmos": 3.0}` (empty = report only) |
| `READ_WER_MAX` | Maximum word error rate between the clip and source readings (empty = report only) |
| `STT_MODULE` | OVOS STT plugin module for the round trip (empty = skip) |
| `STT_CONFIG` | JSON plugin config for `STT_MODULE` |

In [ ]:
stt_module = _env("STT_MODULE").strip()
onnx_runtime = _env("ONNX_RUNTIME", "cpu").strip().lower()

!pip install --quiet --break-system-packages --pre "speakeronnx>=0.0.2a2" "speechonnxmetrics>=0.0.2a2" "librosa>=0.10.2" "soundfile>=0.12.1" "numpy>=1.26" "tqdm>=4.66"

if onnx_runtime == "cuda":
    !pip install --quiet --break-system-packages --pre "onnxruntime-gpu>=1.18"
elif onnx_runtime == "rocm":
    print("AMD ROCm: install onnxruntime-rocm or the MIGraphX build from AMD's package "
          "index yourself; nothing is installed here and CPU is the fallback.")
else:
    !pip install --quiet --break-system-packages --pre "onnxruntime>=1.18"

if stt_module:
    # STT_MODULE names an OVOS STT plugin entrypoint (e.g. ovos-stt-plugin-rover);
    # its PyPI package name is the same string.
    !pip install --quiet --break-system-packages --pre "ovos-plugin-manager>=2.11.6a1"
    !pip install --quiet --break-system-packages --pre {stt_module}
else:
    print("STT_MODULE not set — the round-trip check (CER, reading difference) is skipped.")

In [ ]:
import os
import sys
import csv
import json
import shutil
import re
import unicodedata
import numpy as np
import librosa
import soundfile as sf
from tqdm import tqdm

from speakeronnx import SpeakerEmbedder, cosine
from speechonnxmetrics import score as som_score
from speechonnxmetrics.asr import cer as som_cer


# ═══════════════════════════════════════════════════════════════════
# Text and audio helpers
# ═══════════════════════════════════════════════════════════════════

def _word_error_rate(ref_words: list, hyp_words: list) -> float:
    """Word-level Levenshtein distance divided by the reference length."""
    if not ref_words:
        return 0.0 if not hyp_words else 1.0
    prev = list(range(len(hyp_words) + 1))
    for i, r in enumerate(ref_words, 1):
        cur = [i] + [0] * len(hyp_words)
        for j, h in enumerate(hyp_words, 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (r != h))
        prev = cur
    return prev[-1] / len(ref_words)


def _normalize_for_cer(text: str) -> str:
    """Unicode NFC, casefold, strip punctuation, collapse spaces."""
    text = unicodedata.normalize("NFC", text).casefold()
    text = "".join(" " if unicodedata.category(ch).startswith("P") else ch for ch in text)
    return re.sub(r"\s+", " ", text).strip()


def _envelope(wav: np.ndarray, n_bins: int = 200) -> np.ndarray:
    """Mean-absolute-amplitude envelope in n_bins equal bins, L2-normalised."""
    if len(wav) == 0:
        return np.zeros(n_bins, dtype=np.float32)
    edges = np.linspace(0, len(wav), n_bins + 1).astype(int)
    env = np.array([
        np.mean(np.abs(wav[edges[i]:edges[i + 1]])) if edges[i + 1] > edges[i] else 0.0
        for i in range(n_bins)
    ], dtype=np.float32)
    norm = np.linalg.norm(env)
    return env / norm if norm > 0 else env


def _duration(path: str) -> float:
    info = sf.info(path)
    return info.frames / info.samplerate


def _load_metadata(dataset_dir: str):
    """lj_name -> text, read from metadata.csv (filename|transcription)."""
    rows = []
    path = os.path.join(dataset_dir, "metadata.csv")
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line or "|" not in line:
                continue
            lj_name, text = line.split("|", 1)
            rows.append((lj_name, text))
    return rows


def _load_source_map(dataset_dir: str):
    """lj_name -> source wav filename under DIR_PROCESSED, or {} if absent."""
    mapping = {}
    path = os.path.join(dataset_dir, "source_map.csv")
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line or "|" not in line:
                    continue
                lj_name, wav = line.split("|", 1)
                mapping[lj_name] = wav
    return mapping


def _make_stt(module_name: str, plugin_config: dict, lang: str):
    from ovos_plugin_manager.stt import load_stt_plugin
    cfg = dict(plugin_config)
    cfg.setdefault("lang", lang)
    cls = load_stt_plugin(module_name)
    return cls(config=cfg)


def _transcribe(stt, path: str, lang: str) -> str:
    from ovos_plugin_manager.utils.audio import AudioData
    audio = AudioData.from_file(path)
    return stt.execute(audio, language=lang)


# ═══════════════════════════════════════════════════════════════════
# Per-dataset quality filter
# ═══════════════════════════════════════════════════════════════════

def quality_filter_dataset(
    voice_name: str,
    dataset_dir: str,
    ref_voice_path,
    processed_dir: str,
    get_embedder,
    families: list,
    quality_bars: dict,
    stt,
    lang: str,
    spk_min_cos,
    vc_len_tol_s: float,
    read_wer_max,
    cer_max: float,
    echo_thresh: float,
):
    """
    Score every clip in dataset_dir, move the refused ones to refused/,
    rewrite metadata.csv to the kept rows, and write filter_report.csv.
    Returns (n_kept, n_refused).
    """
    rows = _load_metadata(dataset_dir)
    if not rows:
        print(f"  {voice_name}: no rows in metadata.csv, skipping.")
        return 0, 0

    source_map = _load_source_map(dataset_dir)
    wavs_dir = os.path.join(dataset_dir, "wavs")
    refused_dir = os.path.join(dataset_dir, "refused")

    ref_env = None
    if ref_voice_path:
        ref_wav, _ = librosa.load(ref_voice_path, sr=None, mono=True)
        ref_env = _envelope(ref_wav)

    records = []
    for lj_name, text in tqdm(rows, desc=f"  Filter {voice_name}", unit="clip", file=sys.stdout):
        clip_path = os.path.join(wavs_dir, f"{lj_name}.wav")
        rec = {"lj_name": lj_name, "text": text}
        if not os.path.exists(clip_path):
            rec["refusal_reason"] = "missing_wav"
            records.append(rec)
            continue

        wav, sr = librosa.load(clip_path, sr=None, mono=True)
        rec["duration_s"] = len(wav) / sr

        # Speaker similarity (only when a reference voice exists)
        spk_cos = None
        if ref_voice_path:
            embedder = get_embedder()
            spk_cos = float(cosine(embedder.embed(clip_path), embedder.embed(ref_voice_path)))
        rec["spk_cos"] = spk_cos

        # Conversion length: converted duration against donor source duration
        len_ratio, len_diff_s = None, None
        source_wav = source_map.get(lj_name)
        source_path = os.path.join(processed_dir, source_wav) if source_wav else None
        has_source = bool(source_path and os.path.exists(source_path))
        if ref_voice_path and has_source:
            source_dur = _duration(source_path)
            if source_dur > 0:
                len_ratio = rec["duration_s"] / source_dur
                len_diff_s = rec["duration_s"] - source_dur
        rec["len_ratio"] = len_ratio
        rec["len_diff_s"] = len_diff_s

        # Prompt echo: envelope cosine between the clip and the reference voice
        echo_cos = None
        if ref_env is not None:
            echo_cos = float(np.dot(_envelope(wav), ref_env))
        rec["echo_cos"] = echo_cos

        # Naturalness (no-reference MOS families)
        quality = som_score(clip_path, metrics=families)
        for key, value in quality.items():
            if key != "_errors":
                rec[key] = value

        # Round trip: transcribe the clip and its source with the same STT
        cer_val, read_wer = None, None
        if stt is not None:
            hyp_clip = _normalize_for_cer(_transcribe(stt, clip_path, lang))
            norm_prompt = _normalize_for_cer(text)
            try:
                cer_val = som_cer(norm_prompt, hyp_clip)
            except Exception:
                cer_val = None
            same_file = has_source and os.path.abspath(source_path) == os.path.abspath(clip_path)
            if has_source and not same_file:
                hyp_source = _normalize_for_cer(_transcribe(stt, source_path, lang))
                read_wer = _word_error_rate(hyp_source.split(), hyp_clip.split())
        rec["cer"] = cer_val
        rec["read_wer"] = read_wer

        # Refusal rules, in order: speaker cosine, length difference, CER,
        # reading difference, echo, quality bars.
        reason = ""
        if spk_min_cos is not None and spk_cos is not None and spk_cos < spk_min_cos:
            reason = "speaker_cosine"
        elif len_diff_s is not None and abs(len_diff_s) > vc_len_tol_s + 1e-6:
            reason = "length"
        elif cer_val is not None and cer_val > cer_max:
            reason = "cer"
        elif read_wer_max is not None and read_wer is not None and read_wer > read_wer_max:
            reason = "reading"
        elif echo_cos is not None and echo_cos >= echo_thresh:
            reason = "echo"
        else:
            for bar_key, bar_val in quality_bars.items():
                if rec.get(bar_key) is not None and rec[bar_key] < bar_val:
                    reason = f"quality:{bar_key}"
                    break
        rec["refusal_reason"] = reason
        records.append(rec)

    # ── Split kept / refused, move refused wavs, rewrite metadata.csv ──
    os.makedirs(refused_dir, exist_ok=True)
    kept_rows, refused_rows = [], []
    for rec in records:
        if rec["refusal_reason"]:
            refused_rows.append(rec)
            src = os.path.join(wavs_dir, f"{rec['lj_name']}.wav")
            if os.path.exists(src):
                shutil.move(src, os.path.join(refused_dir, f"{rec['lj_name']}.wav"))
        else:
            kept_rows.append(rec)

    meta_path = os.path.join(dataset_dir, "metadata.csv")
    with open(meta_path, "w", encoding="utf-8") as f:
        f.write("\n".join(f"{r['lj_name']}|{r['text']}" for r in kept_rows) + "\n")

    # ── Per-clip report ────────────────────────────────────────────
    fixed_cols = ["lj_name", "text", "duration_s", "spk_cos", "len_ratio", "len_diff_s", "echo_cos"]
    tail_cols = ["cer", "read_wer", "refusal_reason"]
    metric_cols = sorted({
        k for r in records for k in r
        if k not in fixed_cols and k not in tail_cols
    })
    fieldnames = fixed_cols + metric_cols + tail_cols

    report_path = os.path.join(dataset_dir, "filter_report.csv")
    with open(report_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        for rec in records:
            writer.writerow(rec)

    # ── Per-rule refusal counts ──────────────────────────────────────
    reason_counts = {}
    for rec in refused_rows:
        reason_counts[rec["refusal_reason"]] = reason_counts.get(rec["refusal_reason"], 0) + 1
    print(f"  {voice_name}: {len(kept_rows)} kept, {len(refused_rows)} refused")
    for reason, n in sorted(reason_counts.items()):
        print(f"    refused[{reason}]: {n}")

    # ── Metric distributions ─────────────────────────────────────────
    for col in fixed_cols[2:] + metric_cols + ["cer"]:
        values = [r[col] for r in records if isinstance(r.get(col), (int, float))]
        if not values:
            continue
        arr = np.array(values, dtype=float)
        p5, p50, p95 = np.percentile(arr, [5, 50, 95])
        print(f"    {col:20s} min={arr.min():.4f}  p5={p5:.4f}  p50={p50:.4f}  p95={p95:.4f}  max={arr.max():.4f}")

    echo_values = [r["echo_cos"] for r in records if r.get("echo_cos") is not None]
    if echo_values:
        near = sum(1 for v in echo_values if abs(v - echo_thresh) <= 0.05)
        print(f"    echo: {near} of {len(echo_values)} clips within 0.05 of ECHO_THRESH={echo_thresh}")

    assert len(kept_rows) > 0, f"{voice_name}: quality filter kept 0 clips"
    assert len(kept_rows) + len(refused_rows) == len(rows), (
        f"{voice_name}: kept ({len(kept_rows)}) + refused ({len(refused_rows)}) "
        f"!= input rows ({len(rows)})"
    )
    return len(kept_rows), len(refused_rows)


# ═══════════════════════════════════════════════════════════════════
# Run: filter every dataset built in Stage 4
# ═══════════════════════════════════════════════════════════════════

families = [f.strip() for f in _env("QUALITY_FAMILIES", "utmos,dnsmos").split(",") if f.strip()]
quality_bars = json.loads(_env("QUALITY_BARS", "{}") or "{}")
spk_min_cos_raw = _env("SPK_MIN_COS").strip()
spk_min_cos = float(spk_min_cos_raw) if spk_min_cos_raw else None
vc_len_tol_s = float(_env("VC_LEN_TOL_S", "0.04"))
read_wer_max_raw = _env("READ_WER_MAX").strip()
read_wer_max = float(read_wer_max_raw) if read_wer_max_raw else None
cer_max = float(_env("CER_MAX", "0.10"))
echo_thresh = float(_env("ECHO_THRESH", "0.85"))
stt_module = _env("STT_MODULE").strip()
processed_dir = _env("DIR_PROCESSED")

# The speaker embedder is loaded lazily: a donor_raw-only run never needs it.
_embedder_holder = {}


def _get_embedder():
    if "embedder" not in _embedder_holder:
        print("Loading speaker embedder (wespeaker-resnet34)...")
        _embedder_holder["embedder"] = SpeakerEmbedder(model="wespeaker-resnet34")
    return _embedder_holder["embedder"]


stt = None
if stt_module:
    stt_config = json.loads(_env("STT_CONFIG", "{}") or "{}")
    print(f"Loading STT plugin for the round trip: {stt_module}")
    stt = _make_stt(stt_module, stt_config, LANG)
else:
    print("STT_MODULE not set — the round-trip check (CER, reading difference) is skipped.")

FILTER_RESULTS = {}
for voice_name, ds_dir in BUILT_DATASETS.items():
    ref_voice_path = REF_VOICE_PATHS.get(voice_name)
    print(f"\n[{voice_name}] Quality filter ({'with' if ref_voice_path else 'without'} reference voice)")
    FILTER_RESULTS[voice_name] = quality_filter_dataset(
        voice_name=voice_name,
        dataset_dir=ds_dir,
        ref_voice_path=ref_voice_path,
        processed_dir=processed_dir,
        get_embedder=_get_embedder,
        families=families,
        quality_bars=quality_bars,
        stt=stt,
        lang=LANG,
        spk_min_cos=spk_min_cos,
        vc_len_tol_s=vc_len_tol_s,
        read_wer_max=read_wer_max,
        cer_max=cer_max,
        echo_thresh=echo_thresh,
    )

print(f"\nQuality filter complete for {len(FILTER_RESULTS)} dataset(s).")

---
## 6 · Upload to HuggingFace (Optional)

Pushes each per-voice LJSpeech dataset to HuggingFace.  
Repo names are `{HF_REPO_PREFIX}_{voice_name}` (e.g. `user/tts_alice`).

| Key env vars | |
|---|---|
| `HF_UPLOAD` | `1` to enable |
| `HF_REPO_PREFIX` | e.g. `your-user/tts` — voice name is appended |
| `HF_TOKEN` | Write token (optional if already logged in) |

In [ ]:
if _flag("HF_UPLOAD"):
    !pip install --quiet --break-system-packages --pre "huggingface_hub>=0.34"
else:
    print("HF_UPLOAD not enabled — skipping install.")

In [ ]:
import os

if not _flag("HF_UPLOAD"):
    print("HF_UPLOAD not enabled — skipping upload.")
else:
    repo_prefix = _env("HF_REPO_PREFIX")
    hf_token    = _env("HF_TOKEN") or None

    if not repo_prefix:
        print("ERROR: HF_REPO_PREFIX not set.  Example: your-user/tts")
    else:
        from huggingface_hub import HfApi
        api = HfApi(token=hf_token)

        for voice_name, ds_dir in BUILT_DATASETS.items():
            repo_id = f"{repo_prefix}_{voice_name}"
            print(f"Uploading {voice_name} -> {repo_id} ...")

            api.create_repo(
                repo_id=repo_id,
                repo_type="dataset",
                exist_ok=True,
            )
            api.upload_folder(
                folder_path=ds_dir,
                repo_id=repo_id,
                repo_type="dataset",
                commit_message=f"Upload TTS dataset: {_env('TTS_DATASET_NAME')} / {voice_name}",
            )
            print(f"  Done: https://huggingface.co/datasets/{repo_id}")

        print(f"\nAll {len(BUILT_DATASETS)} dataset(s) uploaded.")

---
## 7 · Output Summary

Quick sanity check: list all output directories and count generated files.

In [ ]:
from pathlib import Path

print("=" * 65)
print("  OUTPUT SUMMARY")
print("=" * 65)

# Pipeline dirs
for label, d in [
    ("Donor TTS (raw)",  _env("DIR_DONOR_TTS")),
    ("Post-processed",   _env("DIR_PROCESSED")),
]:
    p = Path(d)
    if p.exists():
        wav_n = len(list(p.rglob("*.wav")))
        print(f"  {label:25s}  {wav_n:>6} wav files")
    else:
        print(f"  {label:25s}  (not created)")

print(f"{'─' * 65}")
print(f"  {'DATASET':25s}  {'WAVs':>6}  {'meta lines':>10}")
print(f"{'─' * 65}")

# Per-voice datasets
datasets_root = Path(_env("DIR_DATASETS"))
if datasets_root.exists():
    for ds_dir in sorted(datasets_root.iterdir()):
        if not ds_dir.is_dir():
            continue
        wav_n = len(list((ds_dir / "wavs").rglob("*.wav"))) if (ds_dir / "wavs").exists() else 0
        meta = ds_dir / "metadata.csv"
        meta_n = sum(1 for line in open(meta) if line.strip()) if meta.exists() else 0

        marker = "✓" if wav_n > 0 and wav_n == meta_n else "⚠"
        print(f"  {marker} {ds_dir.name:23s}  {wav_n:>6}  {meta_n:>10}")

        # Preview first & last line of metadata
        if meta.exists():
            lines = meta.read_text(encoding="utf-8").strip().splitlines()
            if lines:
                print(f"    first: {lines[0][:90]}{'…' if len(lines[0]) > 90 else ''}")
            if len(lines) > 1:
                print(f"    last : {lines[-1][:90]}{'…' if len(lines[-1]) > 90 else ''}")
else:
    print("  (no datasets created yet)")

print("=" * 65)